# Cypher 기본 문법 단계별 실습
Neo4j Cypher 쿼리 언어 기초 - 제조업 불량 데이터 예제

## 환경 설정

In [1]:
!pip install neo4j -q

from neo4j import GraphDatabase

URI = "neo4j+s://ce6843cb.databases.neo4j.io"  # Aura면 neo4j+s
USER = "ce6843cb"
PASSWORD =  "cNULIOfPxWEBZ1IXFZSUrPcDivYlSrfB1zfGhlz2-zA"

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

with driver.session() as session:    
    print(session.run("RETURN 1 AS ok").single()["ok"])

def run_query(query, params=None):
    with driver.session() as session:
        result = session.run(query, params or {})
        return [dict(r) for r in result]

print('Neo4j 연결 완료')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 28.3 MB/s eta 0:00:00
1
Neo4j 연결 완료


## Step 1 - 노드 만들기: CREATE vs MERGE

In [2]:
# 기존 데이터 초기화
run_query('MATCH (n) DETACH DELETE n')
print('초기화 완료')

초기화 완료


In [3]:
# CREATE - 중복 상관없이 무조건 생성
run_query('''
CREATE (n:설비 {name: "CNC 머신 A-1", location: "1공장"})
''')
print('CREATE 완료')

CREATE 완료


In [4]:
# MERGE - 없으면 생성, 있으면 그냥 사용 (실무에서 주로 사용)
run_query('''
MERGE (n:설비 {name: "CNC 머신 A-1"})
MERGE (m:설비 {name: "프레스 P-2"})
MERGE (k:설비 {name: "용접로봇 W-1"})
''')
print('MERGE 완료')

MERGE 완료


In [5]:
# 생성된 노드 확인
result = run_query('MATCH (n:설비) RETURN n.name AS 설비명')
for r in result:
    print(r)

{'설비명': 'CNC 머신 A-1'}
{'설비명': '프레스 P-2'}
{'설비명': '용접로봇 W-1'}


## Step 2 - 관계 만들기

In [6]:
# 불량유형 노드 추가
run_query('''
MERGE (d1:불량유형 {name: "Burr"})
MERGE (d2:불량유형 {name: "크랙"})
MERGE (d3:불량유형 {name: "치수불량"})
''')

# 설비 → 불량유형 관계 연결
run_query('''
MATCH (eq:설비    {name: "CNC 머신 A-1"})
MATCH (defect:불량유형 {name: "Burr"})
MERGE (eq)-[:발생 {report_id: "RPT-001"}]->(defect)
''')

run_query('''
MATCH (eq:설비    {name: "프레스 P-2"})
MATCH (defect:불량유형 {name: "크랙"})
MERGE (eq)-[:발생 {report_id: "RPT-002"}]->(defect)
''')

print('관계 생성 완료')

관계 생성 완료


## Step 3 - 조회하기: MATCH + RETURN

In [7]:
# 모든 설비 조회
result = run_query('''
MATCH (n:설비)
RETURN n.name AS 설비명
''')
print('=== 전체 설비 ===')
for r in result:
    print(r)

=== 전체 설비 ===
{'설비명': 'CNC 머신 A-1'}
{'설비명': '프레스 P-2'}
{'설비명': '용접로봇 W-1'}


In [8]:
# 설비 → 불량유형 관계 조회
result = run_query('''
MATCH (eq:설비)-[:발생]->(defect:불량유형)
RETURN eq.name AS 설비명, defect.name AS 불량유형
''')
print('=== 설비별 불량유형 ===')
for r in result:
    print(r)

=== 설비별 불량유형 ===
{'설비명': 'CNC 머신 A-1', '불량유형': 'Burr'}
{'설비명': '프레스 P-2', '불량유형': '크랙'}


## Step 4 - 조건 필터: WHERE

In [9]:
# 특정 설비 필터
result = run_query('''
MATCH (eq:설비)-[:발생]->(defect:불량유형)
WHERE eq.name = "CNC 머신 A-1"
RETURN eq.name AS 설비명, defect.name AS 불량유형
''')
print('=== CNC 머신 A-1 불량 현황 ===')
for r in result:
    print(r)

=== CNC 머신 A-1 불량 현황 ===
{'설비명': 'CNC 머신 A-1', '불량유형': 'Burr'}


## Step 5 - UNWIND: 리스트를 행으로 풀기

In [10]:
# 샘플 데이터
reports = [
    {"report_id": "RPT-001", "설비명": "CNC 머신 A-1", "불량유형": "Burr",   "M4분류": "설비",  "담당파트": "설비팀"},
    {"report_id": "RPT-002", "설비명": "프레스 P-2",   "불량유형": "크랙",   "M4분류": "재료",  "담당파트": "품질팀"},
    {"report_id": "RPT-003", "설비명": "용접로봇 W-1", "불량유형": "치수불량","M4분류": "작업",  "담당파트": "생산팀"},
    {"report_id": "RPT-004", "설비명": "CNC 머신 A-1", "불량유형": "치수불량","M4분류": "방법",  "담당파트": "기술팀"},
]

# 기존 데이터 초기화
run_query('MATCH (n) DETACH DELETE n')

# UNWIND로 리스트 일괄 적재
create_query = '''
UNWIND $reports AS report
MERGE (eq:설비       {name: report.설비명})
MERGE (defect:불량유형 {name: report.불량유형})
MERGE (m4:M4분류     {name: report.M4분류})
MERGE (dept:담당파트  {name: report.담당파트})
MERGE (eq)-[:발생     {report_id: report.report_id}]->(defect)
MERGE (defect)-[:원인분류 {report_id: report.report_id}]->(m4)
MERGE (m4)-[:조치담당 {report_id: report.report_id}]->(dept)
'''

run_query(create_query, {"reports": reports})
print(f'{len(reports)}건 적재 완료')

4건 적재 완료


## Step 6 - 전체 그래프 조회

In [11]:
# 3단계 연결 전체 조회
result = run_query('''
MATCH (eq:설비)-[:발생]->(defect:불량유형)-[:원인분류]->(m4:M4분류)-[:조치담당]->(dept:담당파트)
RETURN eq.name AS 설비, defect.name AS 불량유형, m4.name AS M4분류, dept.name AS 담당파트
ORDER BY eq.name
''')

print('=== 전체 불량 현황 ===')
for r in result:
    print(r)

=== 전체 불량 현황 ===
{'설비': 'CNC 머신 A-1', '불량유형': 'Burr', 'M4분류': '설비', '담당파트': '설비팀'}
{'설비': 'CNC 머신 A-1', '불량유형': '치수불량', 'M4분류': '작업', '담당파트': '생산팀'}
{'설비': 'CNC 머신 A-1', '불량유형': '치수불량', 'M4분류': '방법', '담당파트': '기술팀'}
{'설비': '용접로봇 W-1', '불량유형': '치수불량', 'M4분류': '작업', '담당파트': '생산팀'}
{'설비': '용접로봇 W-1', '불량유형': '치수불량', 'M4분류': '방법', '담당파트': '기술팀'}
{'설비': '프레스 P-2', '불량유형': '크랙', 'M4분류': '재료', '담당파트': '품질팀'}


In [12]:
# 설비별 불량 건수 집계
result = run_query('''
MATCH (eq:설비)-[:발생]->(defect:불량유형)
RETURN eq.name AS 설비명, COUNT(defect) AS 불량건수
ORDER BY 불량건수 DESC
''')

print('=== 설비별 불량 건수 ===')
for r in result:
    print(r)

=== 설비별 불량 건수 ===
{'설비명': 'CNC 머신 A-1', '불량건수': 2}
{'설비명': '프레스 P-2', '불량건수': 1}
{'설비명': '용접로봇 W-1', '불량건수': 1}
